# Tutorial 06 — Background Workflows

**No API key required. Fully deterministic.**

eXo-brain can run long-lived workflows as background DAG jobs — tasks with declared
dependencies, automatic retries, and checkpoint-based resume. This tutorial shows:
- How to build a `TaskGraph` (a DAG of async task nodes)
- How to submit and run jobs via `BackgroundRuntime`
- How failures surface as structured `TaskOutcome` objects
- How `retry_limit` makes a flaky node resilient
- How `InMemoryCheckpointStore` enables resume from a mid-job checkpoint

No model calls. No API keys. All async execution is wrapped in `asyncio.run()`.

In [1]:
import pathlib
import sys, os

_repo = pathlib.Path(os.path.abspath(".."))
sys.path.insert(0, str(_repo))
_contracts_src = _repo / "packages" / "eXo_adapters" / "packages" / "exo-brain-core-contracts" / "src"
if _contracts_src.is_dir():
    sys.path.insert(0, str(_contracts_src))

try:
    from dotenv import load_dotenv
    load_dotenv("../.env", override=False)
except ImportError:
    pass

import asyncio
try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

## Part 1 — Build a 4-node DAG

A `TaskGraph` is a Directed Acyclic Graph of `TaskNode` objects.
Each node has a `handler: async def (payload: dict) -> dict` and an optional `depends_on` list.

Pipeline: `fetch → validate → enrich → publish`

In [2]:
from src.core.task_graph import TaskGraph, TaskNode, TaskStatus, TaskOutcome
from src.core.checkpoint_store import InMemoryCheckpointStore
from src.core.worker_pool import WorkerPool
from src.core.scheduler import TaskScheduler, SchedulerResult
from src.core.background_runtime import BackgroundRuntime, BackgroundJob, JobStatus
from src.persistence.contracts import CheckpointRecord, CheckpointStatus
from src.observability.logging import StructuredLogger
from src.observability.metrics import RuntimeMetrics
from src.observability.timeline import RuntimeTimeline

async def fetch_handler(payload: dict) -> dict:
    print("  [fetch]    running...")
    return {"raw_data": [1, 2, 3, 4, 5], "source": "demo"}

async def validate_handler(payload: dict) -> dict:
    print("  [validate] running...")
    deps = payload.get("dependencies", {})
    data = deps.get("fetch", {}).get("raw_data", [])
    assert len(data) > 0, "No data to validate"
    return {"validated": True, "record_count": len(data)}

async def enrich_handler(payload: dict) -> dict:
    print("  [enrich]   running...")
    deps = payload.get("dependencies", {})
    count = deps.get("validate", {}).get("record_count", 0)
    return {"enriched_records": count * 2, "enrichment": "demo_v1"}

async def publish_handler(payload: dict) -> dict:
    print("  [publish]  running...")
    deps = payload.get("dependencies", {})
    enriched = deps.get("enrich", {}).get("enriched_records", 0)
    return {"published": True, "records_published": enriched}

graph = TaskGraph(nodes=[
    TaskNode(node_id="fetch",    handler=fetch_handler),
    TaskNode(node_id="validate", handler=validate_handler, depends_on=["fetch"]),
    TaskNode(node_id="enrich",   handler=enrich_handler,   depends_on=["validate"]),
    TaskNode(node_id="publish",  handler=publish_handler,  depends_on=["enrich"]),
])

print("DAG nodes:", graph.node_ids())

DAG nodes: ['fetch', 'validate', 'enrich', 'publish']


## Part 2 — Run the happy-path job

Wire `TaskScheduler` and `BackgroundRuntime`, submit the graph, and inspect outcomes.

In [3]:
def make_runtime(checkpoint_store=None):
    store = checkpoint_store or InMemoryCheckpointStore()
    pool = WorkerPool(max_concurrency=4)
    logger = StructuredLogger()
    metrics = RuntimeMetrics()
    timeline = RuntimeTimeline()
    scheduler = TaskScheduler(
        worker_pool=pool,
        checkpoint_store=store,
        logger=logger,
        metrics=metrics,
        timeline=timeline,
    )
    runtime = BackgroundRuntime(
        scheduler=scheduler,
        logger=logger,
        metrics=metrics,
        timeline=timeline,
    )
    return runtime, scheduler, store

runtime, scheduler, checkpoint_store = make_runtime()

async def run_job(graph, runtime, job_id="job-happy-001"):
    submitted_id = runtime.submit(graph=graph, payload={}, job_id=job_id)
    # Wait for completion
    for _ in range(200):
        await asyncio.sleep(0.01)
        job = runtime.get_job(submitted_id)
        if job.status in (JobStatus.COMPLETED, JobStatus.FAILED, JobStatus.CANCELLED):
            break
    return runtime.get_job(submitted_id)

job = asyncio.run(run_job(graph, runtime))

print(f"\nJob status: {job.status}")
assert job.status == JobStatus.COMPLETED, f"Expected COMPLETED, got {job.status}"

print("\nNode outcomes:")
for node_id, outcome in job.result.outcomes.items():
    print(f"  {node_id:12} status={outcome.status.value:10} output={outcome.output}")
print("\nPASS — all 4 nodes completed successfully")

  [fetch]    running...
  [validate] running...
  [enrich]   running...
  [publish]  running...

Job status: JobStatus.COMPLETED

Node outcomes:
  fetch        status=completed  output={'raw_data': [1, 2, 3, 4, 5], 'source': 'demo'}
  validate     status=completed  output={'validated': True, 'record_count': 5}
  enrich       status=completed  output={'enriched_records': 10, 'enrichment': 'demo_v1'}
  publish      status=completed  output={'published': True, 'records_published': 10}

PASS — all 4 nodes completed successfully


## Part 3 — Structured failure

When a node raises an exception, execution stops at that node. Downstream nodes are
cancelled. The `TaskOutcome` carries `status=FAILED`, `reason_code`, and `error_message`.

In [4]:
async def failing_validate(payload: dict) -> dict:
    print("  [validate] raising ValueError...")
    raise ValueError("Schema mismatch: field 'id' missing")
graph_fail = TaskGraph(nodes=[
    TaskNode(node_id="fetch",    handler=fetch_handler),
    TaskNode(node_id="validate", handler=failing_validate, depends_on=["fetch"]),
    TaskNode(node_id="enrich",   handler=enrich_handler,   depends_on=["validate"]),
    TaskNode(node_id="publish",  handler=publish_handler,  depends_on=["enrich"]),
])

runtime_fail, _, _ = make_runtime()
job_fail = asyncio.run(run_job(graph_fail, runtime_fail, job_id="job-fail-001"))

print(f"\nJob status: {job_fail.status}")
assert job_fail.status == JobStatus.FAILED

validate_outcome = job_fail.result.outcomes.get("validate")
print(f"validate outcome status : {validate_outcome.status}")
print(f"validate reason_code    : {validate_outcome.reason_code}")
print(f"validate error_message  : {validate_outcome.error_message}")

# Downstream nodes should not have run
enrich_outcome = job_fail.result.outcomes.get("enrich")
if enrich_outcome:
    print(f"enrich status (cancelled/not run): {enrich_outcome.status}")

print("\nPASS — failure is structured; downstream nodes did not run")

  [fetch]    running...
  [validate] raising ValueError...

Job status: JobStatus.FAILED
validate outcome status : TaskStatus.FAILED
validate reason_code    : TASK_EXECUTION_ERROR
validate error_message  : Schema mismatch: field 'id' missing
enrich status (cancelled/not run): TaskStatus.CANCELLED

PASS — failure is structured; downstream nodes did not run


## Part 4 — Retry with a flaky node

`TaskNode(retry_limit=2)` means the node is attempted up to 3 times total
(1 initial + 2 retries). A flaky handler that fails twice then succeeds will
show `outcome.attempts == 3`.

In [5]:
_flaky_call_count = 0

async def flaky_validate(payload: dict) -> dict:
    global _flaky_call_count
    _flaky_call_count += 1
    print(f"  [flaky_validate] attempt {_flaky_call_count}...")
    if _flaky_call_count < 3:
        raise RuntimeError(f"Transient error on attempt {_flaky_call_count}")
    deps = payload.get("dependencies", {})
    data = deps.get("fetch", {}).get("raw_data", [])
    return {"validated": True, "record_count": len(data) if data else 5}

_flaky_call_count = 0  # reset before run

graph_retry = TaskGraph(nodes=[
    TaskNode(node_id="fetch",    handler=fetch_handler),
    TaskNode(node_id="validate", handler=flaky_validate, depends_on=["fetch"], retry_limit=2),
    TaskNode(node_id="enrich",   handler=enrich_handler, depends_on=["validate"]),
    TaskNode(node_id="publish",  handler=publish_handler, depends_on=["enrich"]),
])

runtime_retry, _, _ = make_runtime()
job_retry = asyncio.run(run_job(graph_retry, runtime_retry, job_id="job-retry-001"))

print(f"\nJob status: {job_retry.status}")
assert job_retry.status == JobStatus.COMPLETED, f"Expected COMPLETED, got {job_retry.status}"

validate_outcome = job_retry.result.outcomes["validate"]
print(f"validate attempts : {validate_outcome.attempts}")
assert validate_outcome.attempts == 3, f"Expected 3 attempts, got {validate_outcome.attempts}"
print("\nPASS — flaky node succeeded on attempt 3 (retry_limit=2)")

  [fetch]    running...
  [flaky_validate] attempt 1...
  [flaky_validate] attempt 2...
  [flaky_validate] attempt 3...
  [enrich]   running...
  [publish]  running...

Job status: JobStatus.COMPLETED
validate attempts : 3

PASS — flaky node succeeded on attempt 3 (retry_limit=2)


## Part 5 — Resume from a checkpoint

`InMemoryCheckpointStore` persists node outcomes. When a job is submitted and a checkpoint
for a node already has `status=COMPLETED`, the scheduler seeds the job result with that node's
output and passes it forward to downstream nodes via `dependencies`.

We pre-populate the store with `fetch` already completed, then submit the job — the scheduler
loads the checkpoint and threads the stored output into `validate`'s dependency map.

In [6]:
_fetch_run_count = 0

async def fetch_tracked(payload: dict) -> dict:
    global _fetch_run_count
    _fetch_run_count += 1
    print(f"  [fetch] executing (run #{_fetch_run_count})")
    return {"raw_data": [10, 20, 30], "source": "resumed"}

graph_resume = TaskGraph(nodes=[
    TaskNode(node_id="fetch",    handler=fetch_tracked),
    TaskNode(node_id="validate", handler=validate_handler, depends_on=["fetch"]),
    TaskNode(node_id="enrich",   handler=enrich_handler,   depends_on=["validate"]),
    TaskNode(node_id="publish",  handler=publish_handler,  depends_on=["enrich"]),
])

JOB_ID = "job-resume-001"

# Pre-populate checkpoint store: fetch is already COMPLETED with known output
pre_store = InMemoryCheckpointStore()

async def prepopulate():
    await pre_store.save_checkpoint(CheckpointRecord(
        job_id=JOB_ID,
        node_id="fetch",
        status=CheckpointStatus.COMPLETED,
        tenant_id="default",
        attempt=1,
        payload={"raw_data": [10, 20, 30], "source": "resumed"},
    ))

asyncio.run(prepopulate())

_fetch_run_count = 0  # reset counter

runtime_resume, scheduler_resume, _ = make_runtime(checkpoint_store=pre_store)

async def run_with_store(graph, runtime, job_id):
    submitted_id = runtime.submit(graph=graph, payload={}, job_id=job_id)
    for _ in range(200):
        await asyncio.sleep(0.01)
        job = runtime.get_job(submitted_id)
        if job.status in (JobStatus.COMPLETED, JobStatus.FAILED, JobStatus.CANCELLED):
            break
    return runtime.get_job(submitted_id)

job_resumed = asyncio.run(run_with_store(graph_resume, runtime_resume, JOB_ID))

print(f"\nJob status: {job_resumed.status}")
assert job_resumed.status == JobStatus.COMPLETED, f"Expected COMPLETED, got {job_resumed.status}"

print("\nNode outcomes:")
for node_id, outcome in job_resumed.result.outcomes.items():
    print(f"  {node_id:12} status={outcome.status.value:10} output={outcome.output}")

# Checkpoint output from fetch was threaded into validate's dependencies
validate_out = job_resumed.result.outcomes["validate"].output
assert validate_out.get("record_count") == 3, f"Expected record_count=3, got {validate_out}"
print("\nPASS — checkpoint output seeded into downstream dependency chain")

  [fetch] executing (run #1)
  [validate] running...
  [enrich]   running...
  [publish]  running...

Job status: JobStatus.COMPLETED

Node outcomes:
  fetch        status=completed  output={'raw_data': [10, 20, 30], 'source': 'resumed'}
  validate     status=completed  output={'validated': True, 'record_count': 3}
  enrich       status=completed  output={'enriched_records': 6, 'enrichment': 'demo_v1'}
  publish      status=completed  output={'published': True, 'records_published': 6}

PASS — checkpoint output seeded into downstream dependency chain


## Summary

| Capability | Module | Key API |
|---|---|---|
| DAG definition | `src/core/task_graph` | `TaskGraph`, `TaskNode(depends_on, retry_limit)` |
| Job submission | `src/core/background_runtime` | `BackgroundRuntime.submit()` |
| Job status polling | `src/core/background_runtime` | `BackgroundRuntime.get_job()` |
| Structured outcomes | `src/core/task_graph` | `TaskOutcome(status, reason_code, error_message, attempts)` |
| Retry on failure | `src/core/task_graph` | `TaskNode(retry_limit=N)` |
| Checkpoint-based resume | `src/core/checkpoint_store` | `InMemoryCheckpointStore` + `CheckpointRecord` |

**Key insight:** Failure is structured, not silent. Retries are declarative.
Checkpoints enable resume without re-executing completed nodes — which matters
for expensive or side-effecting tasks.

### Next steps
- **Tutorial 07** — Governance and anomaly detection: detect runaway tenants, manage BYOC fairness

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Full governance lab (story + optional live) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).